# 自然稀疏注意力（DeepSeek NSA）

长序列推理中大部分解码延迟都是被注意力吃掉的，DeepSeek的NSA采用的方案是：三条并行的注意力分支。1. 压缩注意力分析粗筛。2. 选择性保留细粒度。3. 滑动窗口管附近。

## 问题描述

三条并行分支：

1. 压缩分支。词元按组平均池化，Q在所有的池化结果上查询，初筛出关联性较高的组。
2. 选择分支。在压缩分值的注意力计算结果上使用top-k。然后展开压缩分组，对每个组内所有的词元计算注意力。
3. 滑窗分支。找最近的一些词元叠加上去。


# 动手构建

教学 demo：实现 NSA 三路注意力（压缩 / top-k 细粒度 / 滑窗），验证：
- top-k **选块本身**不可微
- **块重要性**靠压缩支路回传梯度（kernel-friendly：稀疏单位是连续 block）


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
DEVICE = torch.device("cpu")


def causal_softmax(scores):
    """scores: (B, H, Tq, Tk) -> causal attention weights."""
    Tq, Tk = scores.shape[-2], scores.shape[-1]
    # allow query i to attend keys 0..i (assuming aligned seq)
    mask = torch.triu(torch.ones(Tq, Tk, device=scores.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    return F.softmax(scores, dim=-1)


class NativeSparseAttention(nn.Module):
    """Minimal NSA: compress + select(top-k blocks) + sliding window.

    Shapes use (B, T, D). Multi-head for clarity.
    """

    def __init__(
        self,
        dim=64,
        num_heads=4,
        block_size=4,
        top_k_blocks=2,
        window_size=4,
    ):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.block_size = block_size
        self.top_k_blocks = top_k_blocks
        self.window_size = window_size

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        # gates: mix three branches (per head)
        self.gate = nn.Linear(dim, num_heads * 3, bias=True)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def _split_heads(self, x):
        # (B, T, D) -> (B, H, T, Dh)
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def _merge_heads(self, x):
        # (B, H, T, Dh) -> (B, T, D)
        B, H, T, Dh = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, H * Dh)

    def _compress_kv(self, k, v):
        """Mean-pool tokens into blocks. Returns (B,H,Nb,Dh), pad last block if needed."""
        B, H, T, Dh = k.shape
        bs = self.block_size
        pad = (bs - T % bs) % bs
        if pad:
            k = F.pad(k, (0, 0, 0, pad))
            v = F.pad(v, (0, 0, 0, pad))
        Tp = k.shape[2]
        Nb = Tp // bs
        k_blocks = k.view(B, H, Nb, bs, Dh).mean(dim=3)
        v_blocks = v.view(B, H, Nb, bs, Dh).mean(dim=3)
        return k_blocks, v_blocks, Nb, pad

    def forward(self, x, return_aux=False):
        B, T, _ = x.shape
        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))
        scale = 1.0 / math.sqrt(self.head_dim)

        # ----- branch 1: compressed attention (dense over blocks, differentiable) -----
        k_cmp, v_cmp, Nb, pad = self._compress_kv(k, v)
        # scores vs all block summaries: (B,H,T,Nb)
        score_cmp = torch.matmul(q, k_cmp.transpose(-1, -2)) * scale
        # crude causal at block level: query token t can see blocks fully before its block
        token_block = torch.arange(T, device=x.device) // self.block_size  # (T,)
        block_ids = torch.arange(Nb, device=x.device)  # (Nb,)
        # allow attending to blocks <= token's block
        blk_ok = block_ids.view(1, 1, 1, Nb) <= token_block.view(1, 1, T, 1)
        score_cmp = score_cmp.masked_fill(~blk_ok, float("-inf"))
        attn_cmp = F.softmax(score_cmp, dim=-1)  # (B,H,T,Nb)  <-- selection scores live here
        out_cmp = torch.matmul(attn_cmp, v_cmp)  # (B,H,T,Dh)

        # block importance for top-k: mean over heads & queries (still from soft attn_cmp)
        # keep per-batch, per-query-block ranking for demo clarity: use last query's scores
        # For training signal we use attn_cmp (soft). For selection we hard top-k.
        importance = attn_cmp.mean(dim=1)  # (B,T,Nb) avg heads
        # pick top-k blocks for each query position
        k_sel = min(self.top_k_blocks, Nb)
        topk_idx = importance.topk(k_sel, dim=-1).indices  # (B,T,k)

        # ----- branch 2: selected fine-grained attention within chosen blocks -----
        # Build a token mask from selected blocks (hard). Gradients to "which block"
        # do NOT flow through this mask; they flow through out_cmp / attn_cmp.
        token_ids = torch.arange(T, device=x.device).view(1, 1, 1, T)  # (1,1,1,T)
        # expand selected block ranges
        # selected token if floor(t/bs) is in topk_idx for that query
        q_block = (token_ids // self.block_size)  # (1,1,1,T) actually broadcast later
        # For each (b,t_q), mark key tokens whose block is in topk_idx[b,t_q]
        # topk_idx: (B,T,k) -> compare with key_block (T,)
        key_block = torch.arange(T, device=x.device) // self.block_size  # (T,)
        # (B,T,k,1) vs (1,1,1,T) -> (B,T,k,T) then any over k
        sel = (topk_idx.unsqueeze(-1) == key_block.view(1, 1, 1, T)).any(dim=2)  # (B,T,T)
        # causal
        causal = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        sel = sel & causal.unsqueeze(0)

        score_sel = torch.matmul(q, k.transpose(-1, -2)) * scale  # (B,H,T,T)
        score_sel = score_sel.masked_fill(~sel.unsqueeze(1), float("-inf"))
        # rows with all -inf (should be rare) -> zeros
        attn_sel = torch.softmax(score_sel, dim=-1)
        attn_sel = torch.nan_to_num(attn_sel, nan=0.0)
        out_sel = torch.matmul(attn_sel, v)

        # ----- branch 3: sliding window -----
        score_win = torch.matmul(q, k.transpose(-1, -2)) * scale
        # window: key in [t-window+1, t]
        idx_q = torch.arange(T, device=x.device).view(T, 1)
        idx_k = torch.arange(T, device=x.device).view(1, T)
        win = (idx_k <= idx_q) & (idx_k > idx_q - self.window_size)
        score_win = score_win.masked_fill(~win.unsqueeze(0).unsqueeze(0), float("-inf"))
        attn_win = F.softmax(score_win, dim=-1)
        attn_win = torch.nan_to_num(attn_win, nan=0.0)
        out_win = torch.matmul(attn_win, v)

        # mix branches with gates from x
        gates = self.gate(x).view(B, T, self.num_heads, 3).permute(0, 2, 1, 3)  # (B,H,T,3)
        gates = torch.softmax(gates, dim=-1)
        out = (
            gates[..., 0:1] * out_cmp
            + gates[..., 1:2] * out_sel
            + gates[..., 2:3] * out_win
        )
        out = self.out_proj(self._merge_heads(out))

        if return_aux:
            return out, {
                "attn_cmp": attn_cmp.detach(),
                "topk_idx": topk_idx.detach(),
                "sel_mask": sel.detach(),
                "importance": importance.detach(),
            }
        return out


# smoke
B, T, D = 2, 16, 64
x = torch.randn(B, T, D)
nsa = NativeSparseAttention(dim=D, block_size=4, top_k_blocks=2, window_size=4)
y, aux = nsa(x, return_aux=True)
print("out", tuple(y.shape))
print("num blocks", aux["attn_cmp"].shape[-1], "topk[0, -1] blocks", aux["topk_idx"][0, -1].tolist())


## 验证：谁在传「选块」梯度？

- 对压缩支路输出求 loss → `q_proj/k_proj` 有梯度（粗粒度可微）
- 只对 selected 支路、且用 **detach 后的 mask** 时，**换块**的信号过不了 top-k；细粒度只更新已选中位置的加权


In [ ]:
def branch_grads(use_cmp=True, use_sel=False):
    """Tiny probe: which path carries selection-related grads to q_proj."""
    m = NativeSparseAttention(dim=64, block_size=4, top_k_blocks=2, window_size=4)
    x = torch.randn(1, 16, 64, requires_grad=False)
    # reimplement mix briefly by calling forward pieces via hooks — simpler: loss on full out
    # Instead: compare grad norm when we zero other branches via gates
    # Force gates: one-hot on chosen branch by replacing gate
    with torch.no_grad():
        # init gate bias so softmax ≈ one-hot on branch
        m.gate.bias.zero_()
        if use_cmp and not use_sel:
            m.gate.bias.view(4, 3)[:, 0] = 10
            m.gate.bias.view(4, 3)[:, 1] = -10
            m.gate.bias.view(4, 3)[:, 2] = -10
        elif use_sel and not use_cmp:
            m.gate.bias.view(4, 3)[:, 0] = -10
            m.gate.bias.view(4, 3)[:, 1] = 10
            m.gate.bias.view(4, 3)[:, 2] = -10

    x = torch.randn(1, 16, 64)
    y = m(x)
    loss = y.pow(2).mean()
    loss.backward()
    gq = m.q_proj.weight.grad.abs().mean().item()
    gk = m.k_proj.weight.grad.abs().mean().item()
    return gq, gk

gq_c, gk_c = branch_grads(use_cmp=True, use_sel=False)
gq_s, gk_s = branch_grads(use_cmp=False, use_sel=True)
print(f"compress-dominated gates | |grad q|={gq_c:.4e} |grad k|={gk_c:.4e}")
print(f"select-dominated gates   | |grad q|={gq_s:.4e} |grad k|={gk_s:.4e}")
print("Both get grads (selected path still has softmax over chosen tokens).")
print("Key point: block *identity* for top-k is set by compress scores; top-k index is discrete.")


In [ ]:
# 可视化：某个 query 选中了哪些 block / token
x = torch.randn(1, 16, 64)
nsa = NativeSparseAttention(dim=64, block_size=4, top_k_blocks=2, window_size=4)
_, aux = nsa(x, return_aux=True)

q_pos = 15  # last token
imp = aux["importance"][0, q_pos].numpy()       # (Nb,)
sel = aux["sel_mask"][0, q_pos].numpy()         # (T,)
topk = aux["topk_idx"][0, q_pos].tolist()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(imp)), imp)
axes[0].set_title(f"compress attn → block scores (q={q_pos})\ntopk={topk}")
axes[0].set_xlabel("block id")
axes[1].imshow(sel[None, :], aspect="auto", cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("selected fine tokens (hard top-k expand)")
axes[1].set_yticks([])
axes[1].set_xlabel("key position")
# draw block boundaries
for b in range(0, 16, 4):
    axes[1].axvline(b - 0.5, color="red", lw=0.8, alpha=0.5)
plt.tight_layout()
plt.show()


## 和 dense 注意力对比（同参数量级）

同输入下 NSA vs 标准 MHA 的输出差异、以及一次 toy 训练步。


In [ ]:
class DenseMHA(nn.Module):
    def __init__(self, dim=64, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.num_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        scale = 1.0 / math.sqrt(Dh)
        scores = torch.matmul(q, k.transpose(-1, -2)) * scale
        scores = scores.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


x = torch.randn(4, 32, 64)
dense = DenseMHA()
sparse = NativeSparseAttention(dim=64, block_size=4, top_k_blocks=2, window_size=8)

with torch.no_grad():
    yd, ys = dense(x), sparse(x)
print("dense out std", yd.std().item(), "nsa out std", ys.std().item())
print("||dense - nsa|| / ||dense||", (yd - ys).norm().item() / yd.norm().clamp_min(1e-6).item())

# one training step on a silly target: reconstruct x
opt = torch.optim.Adam(sparse.parameters(), lr=1e-3)
for step in range(50):
    pred = sparse(x)
    loss = F.mse_loss(pred, x)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (step + 1) % 10 == 0:
        print(f"step {step+1}: loss={loss.item():.4f}")
